# Hotel Booking Cancellation Prediction

In [63]:
%pip install seaborn


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [64]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

**Load Data**

Load Hotel_Booking/hotel_bookings.csv file provided on Brightspace.

In [65]:
df = pd.read_csv("hotel_bookings.csv")
print(df.columns)

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='str')


# 1. Data Pre-processing (25%)


---






**Drop irrelevant columns**

It will significantly reduce the time and effort you need to invest. As a general guideline, columns containing IDs, dates, or irrelevant information are typically considered redundant and offer little value for predictive analysis.

In [22]:
df = df.drop(["reservation_status", "reservation_status_date"], axis=1)

## Unique values

Find out unique values in columns. This will help you in identifying in-consistent data.

In [14]:
for col_name, col_data in df.items():
    print("\n", col_data.value_counts())


 hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64

 is_canceled
0    75166
1    44224
Name: count, dtype: int64

 lead_time
0      6345
1      3460
2      2069
3      1816
4      1715
       ... 
532       1
387       1
458       1
380       1
463       1
Name: count, Length: 479, dtype: int64

 arrival_date_year
2016    56707
2017    40687
2015    21996
Name: count, dtype: int64

 arrival_date_month
August       13877
July         12661
May          11791
October      11160
April        11089
June         10939
September    10508
March         9794
February      8068
November      6794
December      6780
January       5929
Name: count, dtype: int64

 arrival_date_week_number
33    3580
30    3087
32    3045
34    3040
18    2926
21    2854
28    2853
17    2805
20    2785
29    2763
42    2756
31    2741
41    2699
15    2689
27    2664
25    2663
38    2661
23    2621
35    2593
39    2581
22    2546
24    2498
13    2416
16    2405
19    2402
40    2397
2

## 1.1 Missing Values (10%)

Identify and handle missing values.

In [ ]:
print(df.isnull().sum())



print(df.isnull().sum())

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

## 1.2 Removing Inconsistent values and Outliers (10%)

Detecting inconsistencies can be achieved through a variety of methods. Some can be identified by examining unique values within each column, while others may require a solid understanding of the problem domain. Since you might not be an expert in the hotel or hospitality industry, here are some helpful hints:

Hints:

1. Check for incomplete bookings, such as reservations with zero adults, babies, or children.
2. Examine rows with zeros in both 'stays_in_weekend_nights' and 'stays_in_week_nights.'



In [66]:
invalid_bookings = (
    (df['adults'] + df['children'] + df['babies'] == 0) |
    (df['adults'] == 0) |
    (df['stays_in_weekend_nights'] + df['stays_in_week_nights'] == 0)
)

removed = invalid_bookings.sum()
df = df.loc[~invalid_bookings]

print(f"Removed {removed:,} invalid bookings.")
print(f"Remaining bookings: {len(df):,}\n")

Removed 1,048 invalid bookings.
Remaining bookings: 118,342



In [67]:
numerical_columns = df.select_dtypes(include='number').columns
outliers = {}

for col in numerical_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    if IQR == 0:
        continue
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers[col] = df[col].loc[(df[col] < lower_bound) | (df[col] > upper_bound)]

    if len(outliers[col]) == 0:
        continue

    print(f"{col}: {len(outliers[col])} potential outliers ({(len(outliers[col]) / len(df))*100:.2f}%)")

lead_time: 2977 potential outliers (2.52%)
stays_in_weekend_nights: 258 potential outliers (0.22%)
stays_in_week_nights: 3327 potential outliers (2.81%)
adr: 3891 potential outliers (3.29%)
total_of_special_requests: 2850 potential outliers (2.41%)


In [ ]:
for col, values in outliers.items():
    if len(values) > 0:
        print(f"\n{col}")
        print(values.describe())
        print(values.sort_values(ascending=False).head(10))

        if df[col].nunique() <= 20:
            sns.countplot(data=df, x=col)
        else:
            sns.histplot(df[col], bins=50)

        plt.title(f'Distribution of {col}')
        plt.yscale('log')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plt.tight_layout()
        plt.show()

In [68]:
print(df.loc[df['adr'] == df['adr'].max()].T)
print(
    df.loc[df['reserved_room_type'] == 'A', 'adr']
    .describe(percentiles=[0.95, 0.99, 0.999])
)

adr_outlier = df['adr'].idxmax()
print(f"Removing booking with ADR: {df.loc[adr_outlier, 'adr']}")
df = df.drop(adr_outlier)

                                        48515
hotel                              City Hotel
is_canceled                                 1
lead_time                                  35
arrival_date_year                        2016
arrival_date_month                      March
arrival_date_week_number                   13
arrival_date_day_of_month                  25
stays_in_weekend_nights                     0
stays_in_week_nights                        1
adults                                      2
children                                  0.0
babies                                      0
meal                                       BB
country                                   PRT
market_segment                  Offline TA/TO
distribution_channel                    TA/TO
is_repeated_guest                           0
previous_cancellations                      0
previous_bookings_not_canceled              0
reserved_room_type                          A
assigned_room_type                

In [70]:
zero_iqr_columns = [col for col in numerical_columns if col not in outliers]
for col in zero_iqr_columns:
    print(col)

adults
children
babies
is_repeated_guest
previous_cancellations
previous_bookings_not_canceled
booking_changes
days_in_waiting_list
required_car_parking_spaces


## 1.3 Column data type conversion (5%)

All necessary columns should be correctly converted to appropriate data types.


# 2. Exploratory Data Analysis (25%)


---





You've also been provided with examples of valuable insights that could be of interest to hotel management, including:

* Calculating cancellation percentages for City and Resort hotels.
* Identifying the most frequently ordered meal types.
* Determining the number of returning guests.
* Discovering the most booked room types.
* Exploring correlations between room types and cancellations.


For each of these tasks, choose a suitable type of visualisation covered in
the practical sessions, such as:

* Bar graphs
* Pie charts
* Line charts
* Heatmaps

## 2.1. Calculating cancellation percentages for City and Resort hotels.

In [ ]:
# <<< Your code here >>>

## 2.2. Identifying the most frequently ordered meal types.

In [ ]:
# <<< Your code here >>>

## 2.3. Determining the number of returning guests.

In [ ]:
# <<< Your code here >>>

## 2.4. Discovering the most booked room types.

In [ ]:
# <<< Your code here >>>

## 2.5. Exploring correlations between room types and cancellations.

In [ ]:
# <<< Your code here >>>

# 3. Feature Engineering (20%)


---





Apply various feature engineering techniques, covered in the lectures and practicals.

Hint:
* Binning
* Encoding
* Scaling
* Feature selection

In [ ]:
# <<< Your code here >>>

# 4. Classifier Training (20%)


---


Utilise the sklearn Python library to train a ML model (e.g.decision tree classifier). Your process should start with splitting your dataset into input features (X) and a target feature (y). Next, divide the data into 70% training and 30% testing subsets. Train your model on the training dataset and evaluate using test dataset with appropriate metrics. Aim to achieve higher accuracy e.g. more than 70% accuracy using your model.

## 4.1. Data Splitting (5%)

In [ ]:
# <<< Your code here >>>

## 4.2. Model Training (10%)

In [ ]:
# <<< Your code here >>>

## 4.3. Model Evaluation (5%)

In [ ]:
# <<< Your code here >>>

# 5. Feature Importance (10%)


---

Assess the importance of features within your decision tree model. Provide commentary on the reliability of your model's results based on the feature importance scores.

In [ ]:
# <<< Your code here >>>